- Macro HS : 50-10
- Firm HS : 20-20

## **설치**

In [1]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 7.5 MB/s eta 0:00:00


## **설정**

In [2]:
## Google Drive Amount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
## Import libaries
import pandas as pd
import numpy as np

import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.model_selection import GridSearchCV

In [4]:
## Setting
base_path = '/content/drive/MyDrive/CS2/'
file_path = os.path.join(base_path, 'mergedData', '04_M5010_F2020')
output_path = os.path.join(base_path, 'PredData', '04_CatBoost_M5010_F2020')
os.makedirs(output_path, exist_ok=True)

## **데이터 불러오기**

In [5]:
## Load the data
train2018 = pd.read_csv(os.path.join(file_path, 'train2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2018 = pd.read_csv(os.path.join(file_path, 'valid2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2018 = pd.read_csv(os.path.join(file_path, 'test2018.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2019 = pd.read_csv(os.path.join(file_path, 'train2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2019 = pd.read_csv(os.path.join(file_path, 'valid2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2019 = pd.read_csv(os.path.join(file_path, 'test2019.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

train2020 = pd.read_csv(os.path.join(file_path, 'train2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2020 = pd.read_csv(os.path.join(file_path, 'valid2020.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2020 = pd.read_csv(os.path.join(file_path, 'test2020.csv')).sort_values(by=['date', 'gvkey'])

train2021 = pd.read_csv(os.path.join(file_path, 'train2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
valid2021 = pd.read_csv(os.path.join(file_path, 'valid2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)
test2021 = pd.read_csv(os.path.join(file_path, 'test2021.csv')).sort_values(by=['date', 'gvkey']).reset_index(drop=True)

In [6]:
train2018.head()

,ticker,gvkey,permno,sic,exchcd,shrcd,ffi49,ret,date,hs_20_0,...,hs_10_0,hs_10_1,hs_10_2,hs_10_3,hs_10_4,hs_10_5,hs_10_6,hs_10_7,hs_10_8,hs_10_9
0,AVX,1072,81912,3670,1.0,11.0,37,0.101395,1997-01-31,-0.063167,...,-0.014042,-0.02066,0.047901,-0.090232,0.053093,-0.056258,0.084362,-0.070571,0.096853,-0.084651
1,ABS,1240,50032,5411,1.0,11.0,43,-0.013333,1997-01-31,-0.058076,...,-0.014042,-0.02066,0.047901,-0.090232,0.053093,-0.056258,0.084362,-0.070571,0.096853,-0.084651
2,AGREA,1468,13056,2771,3.0,11.0,8,-0.002203,1997-01-31,-0.049929,...,-0.014042,-0.02066,0.047901,-0.090232,0.053093,-0.056258,0.084362,-0.070571,0.096853,-0.084651
3,ASC,1573,44652,5411,1.0,11.0,43,0.027523,1997-01-31,-0.066656,...,-0.014042,-0.02066,0.047901,-0.090232,0.053093,-0.056258,0.084362,-0.070571,0.096853,-0.084651
4,ADM,1722,10516,2070,1.0,11.0,2,-0.102273,1997-01-31,-0.058967,...,-0.014042,-0.02066,0.047901,-0.090232,0.053093,-0.056258,0.084362,-0.070571,0.096853,-0.084651


## **데이터 전처리**

### **datetime**

In [7]:
for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name = f'{df_type}{year}'
        # Ensure the date column is converted to datetime just once for the original dataframes
        if 'date' in globals()[df_name].columns:
            globals()[df_name]['date'] = pd.to_datetime(globals()[df_name]['date'])

### **ticker와 구분 Code 백업 및 X, y분리**

In [8]:
## backup ticker & code and split X, y, and handle date/sasdate columns explicitly

# List of columns to always drop from X dataframes after extraction of Year/Month
# 'date' and 'sasdate' are specifically targeted here as they caused errors.
cols_to_drop_from_X = ['ret', 'date']

for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name = f'{df_type}{year}'
        original_df = globals()[df_name]

        # Backup info columns before any modifications to _X
        globals()[f'{df_name}_info'] = original_df.copy()[['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'date']]

        # Create _X by copying and dropping 'ret', 'date'
        temp_df_X = original_df.copy()
        for col in cols_to_drop_from_X:
            if col in temp_df_X.columns:
                temp_df_X = temp_df_X.drop(col, axis=1)
        globals()[f'{df_name}_X'] = temp_df_X

        # Create _y dataframe
        globals()[f'{df_name}_y'] = original_df.copy()[['ret']]


print("Data split into _X, _y, and _info dataframes, with 'ret', 'date' removed from _X.")

Data split into _X, _y, and _info dataframes, with 'ret', 'date' removed from _X.


In [9]:
## date 컬럼에서 Year, Month 추출 후 문자열로 변환
# 'date' 컬럼은 이미 df_X에서 제거되었으므로, original_df_info에서 가져옵니다.
for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name_X = f'{df_type}{year}_X'
        df_name_info = f'{df_type}{year}_info'

        # Use the 'date' column from the _info dataframe to extract Year and Month
        info_df = globals()[df_name_info]

        # Ensure 'date' in info_df is datetime type (should be from OPM2tS6ciNMA)
        if 'date' in info_df.columns and pd.api.types.is_datetime64_any_dtype(info_df['date']):
            globals()[df_name_X]['Year'] = info_df['date'].dt.year.astype(str)
            globals()[df_name_X]['Month'] = info_df['date'].dt.month.astype(str)
        else:
            # Fallback or error if 'date' is not found or not datetime in info_df
            print(f"Warning: 'date' column not found or not datetime in {df_name_info}. Cannot extract Year/Month.")


print("Year and Month extracted as string types.")

Year and Month extracted as string types.


In [10]:
cols_to_convert_to_str = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49']

# Ensure Year and Month are also treated as string/categorical for CatBoost
# Note: Year and Month are now added in k3CLHCyskNDr, this simply ensures they are string
categorical_features_final = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'Year', 'Month']

for year in range(2018, 2022):
    for df_type in ['train', 'valid', 'test']:
        df_name_X = f'{df_type}{year}_X'
        current_df_X = globals()[df_name_X]

        for col in categorical_features_final:
            if col in current_df_X.columns:
                # Convert to string type, handling potential NaN values which become 'nan'
                current_df_X[col] = current_df_X[col].astype(str)

        globals()[df_name_X] = current_df_X

print("All designated categorical features (including Year and Month) converted to string type in all _X dataframes.")

All designated categorical features (including Year and Month) converted to string type in all _X dataframes.


## **CatBoost**

In [11]:
from catboost import CatBoostRegressor

In [12]:
categorical_features = ['ticker', 'gvkey', 'permno', 'sic', 'exchcd', 'shrcd', 'ffi49', 'Year', 'Month']

catB_2018 = CatBoostRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
catB_2019 = CatBoostRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
catB_2020 = CatBoostRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)
catB_2021 = CatBoostRegressor(random_seed=42, verbose=0, loss_function='RMSE', cat_features=categorical_features)

### **하이퍼파라미터 튜닝**

In [13]:
def tunning_cat(model_base, df_train_X, df_train_y, df_valid_X, df_valid_y, cat_features=None):
    """
    Performs hyperparameter tuning for CatBoostRegressor using a fixed
    training and validation set, without cross-validation.

    Args:
        model_base: An initialized CatBoostRegressor model to use its base parameters
                    (e.g., random_seed, loss_function).
        df_train_X: Training features DataFrame.
        df_train_y: Training target Series/DataFrame.
        df_valid_X: Validation features DataFrame.
        df_valid_y: Validation target Series/DataFrame.
        cat_features: List of categorical feature names.

    Returns:
        tuple: A tuple containing the best trained CatBoostRegressor model
               and a dictionary summarizing the grid search results.
    """

    # 1. Drop 'date' column if present in training or validation features
    #    (assuming 'date' is not a feature for the model)
    if 'date' in df_train_X.columns:
        print("Notice: 'date' column found inside df_train_X. Dropping it now.")
        df_train_X = df_train_X.drop(['date'], axis=1)
    if 'date' in df_valid_X.columns:
        print("Notice: 'date' column found inside df_valid_X. Dropping it now.")
        df_valid_X = df_valid_X.drop(['date'], axis=1)

    # 2. Determine valid categorical features present in the training data
    if cat_features is not None:
        valid_cat_features = [c for c in cat_features if c in df_train_X.columns]
    else:
        # Fallback to an empty list if no categorical features are specified
        valid_cat_features = []

    # 3. Hyperparameter grid for tuning
    param_grid = {
      'iterations': [500],
      'learning_rate': [0.05, 0.1],
      'depth': [6, 8, 10],
      'l2_leaf_reg': [1, 3]
    }

    best_score = float('inf') # Will store RMSE of the best R-squared model
    best_params = None
    best_model_trained = None
    best_r2_score = -float('inf') # Initialize to negative infinity for R-squared (higher is better)

    # Use ParameterGrid from sklearn to generate all combinations
    from sklearn.model_selection import ParameterGrid
    from sklearn.metrics import r2_score # Import r2_score
    grid_search_results_list = [] # Store results for each combination

    print("Starting manual grid search...")
    for i, params_combination in enumerate(ParameterGrid(param_grid)):
        print(f"[{i+1}/{len(ParameterGrid(param_grid))}] Training with params: {params_combination}")

        # Get all parameters from the base model
        base_model_params = model_base.get_params()

        # Create a dictionary for fixed parameters that are NOT part of the tuning grid
        # and ensure no duplication with explicitly passed arguments
        filtered_base_params = {}
        # Include random_seed and loss_function from base_model_params
        if 'random_seed' in base_model_params: # Use get_params() to access parameters
            filtered_base_params['random_seed'] = base_model_params['random_seed']
        if 'loss_function' in base_model_params:
            filtered_base_params['loss_function'] = base_model_params['loss_function']

        # Create a new CatBoostRegressor instance for each parameter combination
        current_model = CatBoostRegressor(
            **filtered_base_params,    # Apply filtered base parameters
            **params_combination,      # Apply current grid parameters
            cat_features=valid_cat_features, # Explicitly set categorical features once
            verbose=0,                 # Suppress verbose output during training loop
        )

        # Train the model with early stopping on the validation set
        current_model.fit(
            df_train_X,
            df_train_y,
            eval_set=(df_valid_X, df_valid_y),
            early_stopping_rounds=50, # Stop if validation error doesn't improve for 50 iterations
            verbose=0 # Suppress verbose output during fit
        )

        # Get the best validation RMSE achieved during training
        # CatBoost stores best metrics in get_best_score()
        current_validation_rmse = current_model.get_best_score()['validation']['RMSE']
        best_iteration = current_model.get_best_iteration()

        # Make predictions on the validation set to calculate R-squared
        valid_predictions = current_model.predict(df_valid_X)
        current_validation_r2 = r2_score(df_valid_y, valid_predictions)

        print(f"  Validation RMSE: {current_validation_rmse:.6f} at iteration {best_iteration}, R-squared: {current_validation_r2:.6f}")

        grid_search_results_list.append({
            'params': params_combination,
            'validation_rmse': current_validation_rmse,
            'validation_r2': current_validation_r2, # Add R-squared
            'best_iteration': best_iteration
        })

        # Prioritize higher R-squared. If R-squared is equal, prioritize lower RMSE.
        if current_validation_r2 > best_r2_score:
            best_r2_score = current_validation_r2
            best_score = current_validation_rmse  # Store RMSE associated with this best R2
            best_params = params_combination
            best_model_trained = current_model
        elif current_validation_r2 == best_r2_score: # If R-squared is equal, use RMSE as a tie-breaker
            if current_validation_rmse < best_score:
                best_score = current_validation_rmse
                best_params = params_combination
                best_model_trained = current_model
                # best_r2_score remains the same as it's a tie

    print(f"\nManual Grid Search Finished.")
    print(f"Best R-squared: {best_r2_score:.6f} (corresponding RMSE: {best_score:.6f}) with parameters: {best_params}")

    # Return the best trained model and a dictionary summarizing the grid search
    # This dictionary mimics the `grid_search_result` structure from the original `model.grid_search`
    result_summary = {
        'best_params': best_params,
        'best_r2_score': best_r2_score, # Best R-squared is the primary metric
        'best_rmse_at_best_r2': best_score, # RMSE corresponding to the best R-squared model
        'all_results': grid_search_results_list
    }

    return best_model_trained, result_summary

In [14]:
print("--- 2018 Training ---")
catB_2018, result_2018 = tunning_cat(catB_2018, train2018_X, train2018_y, valid2018_X, valid2018_y, cat_features=categorical_features)

print("--- 2019 Training ---")
catB_2019, result_2019 = tunning_cat(catB_2019, train2019_X, train2019_y, valid2019_X, valid2019_y, cat_features=categorical_features)

print("--- 2020 Training ---")
catB_2020, result_2020 = tunning_cat(catB_2020, train2020_X, train2020_y, valid2020_X, valid2020_y, cat_features=categorical_features)

print("--- 2021 Training ---")
catB_2021, result_2021 = tunning_cat(catB_2021, train2021_X, train2021_y, valid2021_X, valid2021_y, cat_features=categorical_features)

--- 2018 Training ---
Starting manual grid search...
[1/12] Training with params: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.05}
  Validation RMSE: 0.084098 at iteration 0, R-squared: -0.004086
[2/12] Training with params: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
  Validation RMSE: 0.084308 at iteration 0, R-squared: -0.009105
[3/12] Training with params: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.05}
  Validation RMSE: 0.084095 at iteration 0, R-squared: -0.004011
[4/12] Training with params: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
  Validation RMSE: 0.084290 at iteration 0, R-squared: -0.008673
[5/12] Training with params: {'depth': 8, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.05}
  Validation RMSE: 0.084059 at iteration 1, R-squared: -0.003152
[6/12] Training with params: {'depth': 8, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
  Validation 

### **최종 학습**

In [15]:
# 최적 하이퍼파라미터 튜닝한 모델에
# Train+Valid 합쳐서 최종 학습

for year in range(2018, 2022):
    print(f"--- Final Training for {year} ---")

    # Combine train and valid datasets for the current year
    combined_train_X = pd.concat([globals()[f'train{year}_X'], globals()[f'valid{year}_X']], ignore_index=True)
    combined_train_y = pd.concat([globals()[f'train{year}_y'], globals()[f'valid{year}_y']], ignore_index=True)

    # Retrieve best parameters for the current year from the result dictionary
    # The result dictionary name is 'result_YYYY'
    result_dict_name = f'result_{year}'
    best_params = globals()[result_dict_name]['best_params']

    # Initialize a new CatBoostRegressor model with the best parameters
    # and other fixed parameters (random_seed, loss_function, categorical_features)
    final_model = CatBoostRegressor(
        random_seed=42,
        loss_function='RMSE',
        cat_features=categorical_features,
        verbose=0, # Keep training silent
        **best_params # Unpack the best hyperparameters
    )

    # Train the final model on the combined dataset
    final_model.fit(combined_train_X, combined_train_y, verbose=0)

    # Assign the trained model back to its respective variable (catB_2018, catB_2019, etc.)
    globals()[f'catB_{year}'] = final_model

    print(f"Final model for {year} trained with best parameters: {best_params}")

print("All final models trained successfully on combined train+valid data.")

--- Final Training for 2018 ---
Final model for 2018 trained with best parameters: {'depth': 10, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
--- Final Training for 2019 ---
Final model for 2019 trained with best parameters: {'depth': 6, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.05}
--- Final Training for 2020 ---
Final model for 2020 trained with best parameters: {'depth': 10, 'iterations': 500, 'l2_leaf_reg': 1, 'learning_rate': 0.1}
--- Final Training for 2021 ---
Final model for 2021 trained with best parameters: {'depth': 8, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.1}
All final models trained successfully on combined train+valid data.


## **예측**

In [16]:
def make_result_table(model, test_info, test_X, test_y):
  pred_y = model.predict(test_X)

  result_df = test_info[['ticker', 'date', 'permno']].copy()
  result_df['pred_ret']=pred_y
  result_df['true_ret']=test_y

  return result_df

In [17]:
result_2018 = make_result_table(catB_2018, test2018_info, test2018_X, test2018_y)
result_2019 = make_result_table(catB_2019, test2019_info, test2019_X, test2019_y)
result_2020 = make_result_table(catB_2020, test2020_info, test2020_X, test2020_y)
result_2021 = make_result_table(catB_2021, test2021_info, test2021_X, test2021_y)

In [18]:
result_2018.to_csv(os.path.join(output_path, 'result_2018.csv'), index=False)
result_2019.to_csv(os.path.join(output_path, 'result_2019.csv'), index=False)
result_2020.to_csv(os.path.join(output_path, 'result_2020.csv'), index=False)
result_2021.to_csv(os.path.join(output_path, 'result_2021.csv'), index=False)

In [19]:
result_2018

,ticker,date,permno,pred_ret,true_ret
0,AAL,2018-01-31,21020,-0.051838,0.044013
1,PNW,2018-01-31,27991,-0.025387,-0.053240
2,AAN,2018-01-31,10517,-0.009836,0.026098
3,ABT,2018-01-31,20482,-0.057399,0.094095
4,AMD,2018-01-31,61241,0.006676,0.336576
...,...,...,...,...,...
11560,CBRE,2018-12-31,90199,-0.010224,-0.083333
11561,WCG,2018-12-31,90272,-0.033507,-0.073721
11562,BLKB,2018-12-31,90276,-0.014007,-0.141297
11563,ALNY,2018-12-31,90178,0.005570,-0.101651
